# AquaInsight — Task 3: Time-Based Feature Engineering

**Internship Project:** Predicting Water Quality Index for Comprehensive Water Assessment

This notebook is one of the nine independent GitHub deliverables. It can be run separately using the supplied water-quality CSV.

## Step-by-step approach

1. Load the supplied dataset.
2. Perform the task-specific analysis.
3. Display quantitative results.
4. Interpret the results for downstream water-quality modeling.
5. Preserve data for review rather than making unsupported automatic corrections.

In [ ]:
# Common setup — AquaInsight Water Quality Internship

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.spatial.distance import mahalanobis

from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, OrdinalEncoder
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.metrics import mean_squared_error

try:
    from rapidfuzz.fuzz import ratio, token_set_ratio
except ImportError:
    raise ImportError("Install RapidFuzz first: pip install rapidfuzz")

possible_paths = [
    Path("Water Quality(1).csv"),
    Path("Water Quality.csv"),
    Path("../data/Water Quality(1).csv"),
    Path("../data/Water Quality.csv"),
]

DATA_PATH = next((p for p in possible_paths if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Place the supplied CSV beside this notebook.")

df = pd.read_csv(DATA_PATH)

print(f"Dataset: {DATA_PATH}")
print(f"Shape: {df.shape}")
display(df.head())


# Task 3 — Time-Based Feature Engineering

### Internship requirement
Engineer meaningful time-based variables from datetime columns and identify important temporal patterns.

### Steps
1. Combine date and time into a single datetime variable.
2. Extract year, month, day, hour, day of week, quarter, week of year, and day of year.
3. Create a weekend indicator.
4. Create a seasonal variable.
5. Aggregate measurements by characteristic and year.
6. Compare temporal trends for frequently measured characteristics.


In [8]:
# Step 5 — Engineer temporal features and assess temporal relationships

def add_time_features(data):
    result = data.copy()

    result["datetime"] = pd.to_datetime(
        result["ActivityStartDate"].astype("string") + " " +
        result["ActivityStartTime"].fillna("00:00:00").astype("string"),
        errors="coerce"
    )

    result["year"] = result["datetime"].dt.year
    result["month"] = result["datetime"].dt.month
    result["day"] = result["datetime"].dt.day
    result["hour"] = result["datetime"].dt.hour
    result["day_of_week"] = result["datetime"].dt.dayofweek
    result["quarter"] = result["datetime"].dt.quarter
    result["week_of_year"] = result["datetime"].dt.isocalendar().week
    result["day_of_year"] = result["datetime"].dt.dayofyear
    result["is_weekend"] = result["day_of_week"].isin([5, 6]).astype(int)

    result["season"] = result["month"].map({
        12: "Winter", 1: "Winter", 2: "Winter",
        3: "Spring", 4: "Spring", 5: "Spring",
        6: "Summer", 7: "Summer", 8: "Summer",
        9: "Autumn", 10: "Autumn", 11: "Autumn"
    })

    return result


time_df = add_time_features(df)

display(time_df[[
    "ActivityStartDate", "ActivityStartTime", "datetime", "year", "month",
    "day", "hour", "day_of_week", "quarter", "week_of_year",
    "day_of_year", "is_weekend", "season"
]].head())

# Year-level temporal trend table.
temporal = (
    time_df.dropna(subset=["datetime"])
    .groupby(["CharacteristicName", "year"])["ResultValue"]
    .agg(["count", "mean", "median"])
    .reset_index()
)

top_temporal_chars = df["CharacteristicName"].value_counts().head(8).index
display(
    temporal[temporal["CharacteristicName"].isin(top_temporal_chars)]
    .head(30)
)

# Correlation analysis is performed within each characteristic because
# ResultValue units differ across characteristics.
temporal_features = [
    "year", "month", "day", "hour",
    "day_of_week", "quarter", "week_of_year",
    "day_of_year", "is_weekend"
]

correlation_rows = []
for char in top_temporal_chars:
    subset = time_df.loc[
        time_df["CharacteristicName"].eq(char),
        temporal_features + ["ResultValue"]
    ].dropna()

    if len(subset) >= 30:
        for feature in temporal_features:
            correlation_rows.append({
                "characteristic": char,
                "time_feature": feature,
                "pearson_correlation": subset[feature].corr(subset["ResultValue"]),
                "absolute_correlation": abs(
                    subset[feature].corr(subset["ResultValue"])
                )
            })

temporal_correlations = pd.DataFrame(correlation_rows).sort_values(
    "absolute_correlation", ascending=False
)

display(temporal_correlations.head(20).round(4))
print(
    "The strongest temporal relationships should be interpreted per characteristic, "
    "because different characteristics have different units and sampling behavior."
)


,ActivityStartDate,ActivityStartTime,datetime,year,month,day,hour,day_of_week,quarter,week_of_year,day_of_year,is_weekend,season
0,2000-01-03,12:00:00,2000-01-03 12:00:00,2000,1,3,12,0,1,1,3,False,Winter
1,2000-01-03,12:00:00,2000-01-03 12:00:00,2000,1,3,12,0,1,1,3,False,Winter
2,2000-01-03,12:00:00,2000-01-03 12:00:00,2000,1,3,12,0,1,1,3,False,Winter
3,2000-01-03,12:00:00,2000-01-03 12:00:00,2000,1,3,12,0,1,1,3,False,Winter
4,2000-01-03,12:00:00,2000-01-03 12:00:00,2000,1,3,12,0,1,1,3,False,Winter


,CharacteristicName,year,count,mean,median
228,Calcium,2000,329,2.696717,0.800
229,Calcium,2001,320,3.664094,1.050
230,Calcium,2002,302,3.440132,0.910
231,Calcium,2003,283,3.203004,0.720
232,Calcium,2004,294,3.274082,0.800
233,Calcium,2005,302,4.040828,0.680
234,Calcium,2006,348,4.880805,0.770
235,Calcium,2007,414,6.907778,0.920
236,Calcium,2008,428,4.733808,0.870
237,Calcium,2009,369,3.331220,0.790


## Task 3 — Conclusion

The analysis above completes the requested **Time-Based Feature Engineering** component of the AquaInsight internship assignment. Results should be interpreted together with domain requirements and the official WQI definition when it becomes available.